In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import anndata as ad
import datetime
import traceback
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.sparse import csr_matrix
import re

# Input file path (make sure using most recent aligned anndatas)
GE_INPUT = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/102025/ge_adata_matched_2025-10-01.h5ad"
SPLICE_INPUT = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/102025/splice_adata_matched_2025-10-01.h5ad"

# Read the input file
ge_adata = ad.read_h5ad(GE_INPUT)
splice_adata = ad.read_h5ad(SPLICE_INPUT)

# Check the number of cells and genes
print(f"Number of cells: {ge_adata.n_obs}")
print(f"Number of genes: {ge_adata.n_vars}")

In [ ]:
print(splice_adata.shape)
print(ge_adata.shape)

### Types of QC to do before saving final splicing and GE anndatas....
- Gene expression based QC
- Splicing based QC, ensure reasonable number of junctions detected in cells... 
- Calculate nuclear vs cytoplasmic score...

In [ ]:
# Clean up GE gene info...
gene_info_df = pd.read_csv("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/processed_data/gene_info_df_2025-09-30.csv")
gene_info_df["gene_symbol"] = gene_info_df["gene_name"]
ge_adata.var["gene_symbol"] = ge_adata.var.index.values
ge_adata.var_names = ge_adata.var["gene_symbol"]

# reset the index
ge_adata.var = ge_adata.var.reset_index(drop=True)
ge_adata.var = ge_adata.var.merge(gene_info_df, on="gene_symbol", how="left")
splice_adata.var["clean_gene_id"] = splice_adata.var["gene_id"].str.split(".").str[0]
splice_adata.var.drop(columns=["gene_name"], inplace=True)
ge_adata.var["clean_gene_id"] = ge_adata.var["gene_id"].str.split(".").str[0]

print("Clean up cell ids...")
# Confirm that the cell ids are the same
splice_adata.obs_names = splice_adata.obs["cell_id_clean"]
ge_adata.obs_names = ge_adata.obs["cell_id_clean"]
# Assert that obs_names are the same
assert np.all(ge_adata.obs_names == splice_adata.obs_names)
# Assert that the cells are in the same order
assert np.all(ge_adata.obs["cell_id_clean"].values == splice_adata.obs["cell_id_clean"].values)

In [ ]:
# how many genes in common between splicing and gene expression to start with? 
gene_info = ge_adata.var[["clean_gene_id", "gene_name", "mean_transcript_length", "mean_intron_length", "num_transcripts"]]
# remove index column
gene_info = gene_info.reset_index(drop=True)

### Quality-control pipeline  (overview)

1. **Flag rRNA genes**  
   * Parse the comma-separated `transcript_biotypes` column.  
   * Mark any entry that contains `rRNA` or `rRNA_pseudogene` → `is_rRNA = True`.

2. **Per-cell QC metrics** (`scanpy.pp.calculate_qc_metrics`)  
   * Input layer: raw counts  
   * Extra QC var: `is_rRNA` → adds `total_counts_is_rRNA` and `pct_counts_is_rRNA`.

3. **Batch-aware outlier detection**  
   For each `batch` separately:  
   * Compute the **median** and **MAD** (median-absolute-deviation) of  
     * library size (`total_counts`)  
     * number of detected genes (`n_genes_by_counts`) using raw counts.
   * Convert both features to robust z-scores  
     \[
       z = \frac{x-\text{median}}{\text{MAD}\times1.4826}
     \]
     (1.4826 rescales MAD to σ for a normal distribution).

4. **Cell filtering rules**  
   * │z│ > 3 for *either* library size *or* genes detected → drop  
   * `pct_counts_is_rRNA` ≥ 10 % → drop  
   * missing `broad_cell_type` annotation → drop  

5. **Result**  
   After filtering **80,904** cells remain (from initial set of 88,518).

6. **Feature selection for integration**  
   From the `predicted_log_norm_tms` layer, keep the **10 000** most highly variable genes for all downstream analyses.


In [ ]:
print("=== FILTERING CELL TYPES WITH <50 CELLS ===")

# Count cells per broad_cell_type
cell_type_counts = ge_adata.obs['broad_cell_type'].value_counts()
print(f"Cell type counts before filtering:")
print(cell_type_counts.sort_values(ascending=False))

# Keep only cell types with at least 50 cells
valid_cell_types = cell_type_counts[cell_type_counts >= 50].index
print(f"\nCell types with ≥50 cells: {len(valid_cell_types)}")
print(f"Cell types being removed: {len(cell_type_counts) - len(valid_cell_types)}")

# Filter both datasets
cells_before = ge_adata.n_obs
cell_mask = ge_adata.obs['broad_cell_type'].isin(valid_cell_types)
ge_adata = ge_adata[cell_mask].copy()
splice_adata = splice_adata[cell_mask].copy()

print(f"Filtered from {cells_before} to {ge_adata.n_obs} cells")
print(f"Final cell type counts:")
print(ge_adata.obs['broad_cell_type'].value_counts().sort_values(ascending=False))

# Verify alignment
assert np.all(ge_adata.obs_names == splice_adata.obs_names)
print("✓ Datasets remain aligned after cell type filtering")
print(splice_adata.shape), print(ge_adata.shape)

In [ ]:
# ---------- 0   Tag rRNA genes once ---------------------------------
# non-capturing groups (?: … ) silence the pandas warning
rrna_regex = r'(?:^|,\s*)rRNA(?:_pseudogene)?(?:$|,\s*)'
ge_adata.var['is_rRNA'] = ge_adata.var['gene_biotype'].str.contains(
    rrna_regex, regex=True, na=False
)

# ---------- 1   All QC metrics in a single scanpy call --------------
sc.pp.calculate_qc_metrics(
    ge_adata,
    qc_vars=['is_rRNA'],               # adds total_counts_is_rRNA & pct_counts_is_rRNA
    layer='raw_counts',
    percent_top=None,
    inplace=True
)

# ---------- 2   Batch-wise robust stats (median ± MAD) --------------
stats = (
    ge_adata.obs
      .groupby('batch')
      .agg(
          n_genes_median = ('n_genes_by_counts', 'median'),
          n_genes_mad    = ('n_genes_by_counts',
                            lambda x: np.median(np.abs(x - np.median(x)))),
          counts_median  = ('total_counts',       'median'),
          counts_mad     = ('total_counts',
                            lambda x: np.median(np.abs(x - np.median(x))))
      )
)
qc = ge_adata.obs.join(stats, on='batch')

sf = 1.4826     # converts MAD -> SD for a normal distribution
z_genes  = (qc['n_genes_by_counts'] - qc['n_genes_median']) / (sf * qc['n_genes_mad'])
z_counts = (qc['total_counts']       - qc['counts_median'])  / (sf * qc['counts_mad'])

gene_filter   = (-3 <= z_genes)  & (z_genes  <= 3)        # keep within ±3 "robust SD"
count_filter  = (-3 <= z_counts) & (z_counts <= 3)
rrna_filter   =  qc['pct_counts_is_rRNA'] < 10
label_filter  =  qc['broad_cell_type'].notna()

keep = gene_filter & count_filter & rrna_filter & label_filter
ge_adata = ge_adata[keep].copy()
print(f"Retained {ge_adata.n_obs} cells")

print(splice_adata.shape), print(ge_adata.shape)

### Now remove any lncRNAs and then subset by common genes with splice_adata and then mark HVGs

In [ ]:
# Step 1: Subset splice_adata to genes present in gene_info
valid_genes = set(gene_info["clean_gene_id"])
keep_junctions = splice_adata.var["clean_gene_id"].isin(valid_genes)
splice_adata = splice_adata[:, keep_junctions].copy()

# Step 2: Merge splice_adata.var with gene_info
# This will add new columns to .var based on clean_gene_id
splice_adata.var = splice_adata.var.merge(
    gene_info, 
    on="clean_gene_id")

splice_adata.var

In [ ]:
print("=== STEP 1: Remove lncRNAs from ge_adata ===")

# Define lncRNA biotype keywords
lncrna_keywords = ["lncRNA", "lincRNA"]

# Identify lncRNA genes
lncrna_mask = ge_adata.var["gene_biotype"].str.contains(
    "|".join(lncrna_keywords), case=False, na=False
)
print(f"Removing {lncrna_mask.sum():,} lncRNA genes.")
ge_adata = ge_adata[:, ~lncrna_mask].copy()

print("=== STEP 2: Subset to genes shared with splice_adata ===")

# Compute gene overlap
splice_genes = set(splice_adata.var["clean_gene_id"])
ge_genes = set(ge_adata.var["clean_gene_id"])
common_genes = splice_genes & ge_genes

print(f"Keeping {len(common_genes):,} genes shared between GE and Splicing data.")

# Subset ge_adata to shared genes
ge_adata = ge_adata[:, ge_adata.var["clean_gene_id"].isin(common_genes)].copy()

# Also subset splice_adata to shared genes 
splice_adata = splice_adata[:, splice_adata.var["clean_gene_id"].isin(common_genes)].copy()


In [ ]:
# ----------------------------------------------------------------

print("=== STEP 4: Align cells between ge_adata and splice_adata ===")
# Realign splice_adata to match filtered ge_adata cells
splice_adata = splice_adata[ge_adata.obs_names].copy()

# Now safe to compare
assert np.all(ge_adata.obs_names == splice_adata.obs_names)
print(f"✓ Gene expression data filtered to {ge_adata.n_vars} genes")


In [ ]:
# ----------------------------------------------------------------
print("=== STEP 5: Summarize number of junction reads per cell ===")
print("=== CALCULATING JUNCTION QC METRICS ===")

# Confirm required column exists
assert "annotation_status" in splice_adata.var.columns, "Missing 'annotation_status' column in splice_adata.var"
assert "both" in splice_adata.var["annotation_status"].unique(), "'both' not found in annotation_status"

# Define annotation mask
is_annotated = splice_adata.var["annotation_status"] == "both"
is_unannotated = ~is_annotated

# Read count matrix
X_counts = splice_adata.layers["cell_by_junction_matrix"]

# Total read counts
splice_adata.obs["total_junction_reads"] = np.asarray(X_counts.sum(axis=1)).flatten()
splice_adata.obs["annotated_junction_reads"] = np.asarray(X_counts[:, is_annotated].sum(axis=1)).flatten()
splice_adata.obs["unannotated_junction_reads"] = np.asarray(X_counts[:, is_unannotated].sum(axis=1)).flatten()

# Detected junctions (binary matrix)
X_detected = X_counts > 0
splice_adata.obs["n_detected_annotated_junctions"] = np.asarray(X_detected[:, is_annotated].sum(axis=1)).flatten()
splice_adata.obs["n_detected_unannotated_junctions"] = np.asarray(X_detected[:, is_unannotated].sum(axis=1)).flatten()

# Apply filters based on 1st and 99th percentiles
def quantile_filter(series, lower=0.01, upper=0.99):
    q_low, q_high = series.quantile([lower, upper])
    return (series >= q_low) & (series <= q_high)

read_filter = quantile_filter(splice_adata.obs["total_junction_reads"])
junctions_filter = quantile_filter(splice_adata.obs["n_detected_annotated_junctions"])

# Additional filter: cell must have at least 100 total junction reads
min_reads_filter = splice_adata.obs["total_junction_reads"] > 1000
min_junctions_filter = splice_adata.obs["n_detected_annotated_junctions"] >= 2

# Final per-cell QC: within percentile range AND above minimum count
splice_qc_filter = read_filter & junctions_filter & min_reads_filter & min_junctions_filter

# Filter both datasets
cells_before_filter = splice_adata.shape[0]
ge_adata = ge_adata[splice_qc_filter].copy()
splice_adata = splice_adata[splice_qc_filter].copy()
cells_after_filter = splice_adata.shape[0]
 
print(f"Cells lost due to junction-based per-cell QC filters: {cells_before_filter - cells_after_filter}")

print("✓ Applied junction-based per-cell QC filters")

# ----------------------------------------------------------------

print("=== APPLYING JUNCTION-LEVEL QC FILTERS ===")

# Recompute total read counts per junction
junction_counts = np.asarray(splice_adata.layers["cell_by_junction_matrix"].sum(axis=0)).flatten()
junction_detected = np.asarray((splice_adata.layers["cell_by_junction_matrix"] > 0).sum(axis=0)).flatten()

# Annotate junctions based on detection across cells
detected_in_cells = np.asarray((X_counts > 0).sum(axis=0)).flatten()
splice_adata.var["n_cells_detected"] = detected_in_cells
splice_adata.var["confidence"] = np.where(
    splice_adata.var["n_cells_detected"] <= 100, "low", "high"
)
print(f"✓ Annotated {np.sum(splice_adata.var['confidence'] == 'low')} junctions as 'low confidence'")

# Remove junction-ATSEs that have more than 10 junctions suggesting overly complex splicing
junctions_before = splice_adata.shape[1]
atses_before = splice_adata.var["gene_name"].nunique()
splice_adata = splice_adata[:, splice_adata.var["num_junctions"] <= 10]
junctions_after = splice_adata.shape[1]
atses_after = splice_adata.var["gene_name"].nunique()

print(f"✓ Removed {junctions_before - junctions_after} junctions from {atses_before - atses_after} genes/ATSEs")
print(f"  (ATSEs with >10 junctions removed due to overly complex splicing patterns)")
print(f"  Remaining: {junctions_after} junctions across {atses_after} genes/ATSEs")

In [ ]:
# Let's clean up our tissues and cell types 
# Get value counts
cell_counts = splice_adata.obs[["tissue", "cell_type", "broad_cell_type"]].value_counts()

# Filter out combinations that appear only once
cell_counts_filtered = cell_counts[cell_counts > 1]

print(f"Before filtering: {len(cell_counts)} unique combinations")
print(f"After filtering: {len(cell_counts_filtered)} unique combinations")
print(f"Removed {len(cell_counts) - len(cell_counts_filtered)} singleton combinations")

# Now filter the actual AnnData object to remove these singleton cells
# Create a tuple column for the combination
splice_adata.obs['temp_combo'] = list(zip(
    splice_adata.obs['tissue'],
    splice_adata.obs['cell_type'],
    splice_adata.obs['broad_cell_type']
))

# Get the combinations that have more than 1 cell
valid_combos = set(cell_counts_filtered.index)

# Filter to keep only cells in valid combinations
cells_before = splice_adata.shape[0]
splice_adata = splice_adata[splice_adata.obs['temp_combo'].isin(valid_combos)].copy()
cells_after = splice_adata.shape[0]

# Remove the temporary column
splice_adata.obs.drop(columns=['temp_combo'], inplace=True)

print(f"\nCells removed: {cells_before - cells_after}")
print(f"Cells remaining: {cells_after}")

In [ ]:
# To visualize the full filtered table
print("\n=== Full filtered cell type distribution ===")
cell_counts_filtered_display = splice_adata.obs[["tissue", "cell_type", "broad_cell_type"]].value_counts()
print(cell_counts_filtered_display.to_string())  # .to_string() shows the full table without truncation

In [ ]:
import pandas as pd
import numpy as np

def create_medium_cell_type(row):
    """
    Create medium-level cell type classification between broad and granular
    """
    cell_type = row['cell_type']
    broad_type = row['broad_cell_type']
    
    # Excitatory Neurons - keep layer/region info
    if broad_type == 'Excitatory_Neuron':
        if any(marker in cell_type for marker in ['L2', 'L3', 'L4', 'L5', 'L6']):
            # Extract layer info
            for layer in ['L2', 'L3', 'L4', 'L5', 'L6', 'L5/6']:
                if layer in cell_type:
                    if 'IT' in cell_type:
                        return f'{layer} IT Neuron'
                    elif 'CT' in cell_type:
                        return f'{layer} CT Neuron'
                    elif 'PT' in cell_type:
                        return f'{layer} PT Neuron'
                    elif 'ET' in cell_type:
                        return f'{layer} ET Neuron'
                    elif 'NP' in cell_type:
                        return f'{layer} NP Neuron'
                    else:
                        return f'{layer} Excitatory Neuron'
        elif any(region in cell_type for region in ['CA1', 'CA2', 'CA3', 'DG']):
            # Hippocampal regions
            for region in ['CA1', 'CA2', 'CA3', 'DG']:
                if region in cell_type:
                    return f'{region} Excitatory Neuron'
        elif 'IT' in cell_type:
            return 'IT Excitatory Neuron'
        elif 'CT' in cell_type:
            return 'CT Excitatory Neuron'
        else:
            return 'Excitatory Neuron'
    
    # Inhibitory Neurons - keep marker info
    elif broad_type == 'Inhibitory_Neuron':
        if 'SST' in cell_type or 'Sst' in cell_type:
            return 'SST Inhibitory Neuron'
        elif 'PVALB' in cell_type or 'Pvalb' in cell_type:
            return 'PVALB Inhibitory Neuron'
        elif 'VIP' in cell_type or 'Vip' in cell_type:
            return 'VIP Inhibitory Neuron'
        elif 'LAMP5' in cell_type or 'Lamp5' in cell_type:
            return 'LAMP5 Inhibitory Neuron'
        else:
            return 'Inhibitory Neuron'
    
    # GI Epithelial - keep cell type detail
    elif broad_type == 'GI_Epithelial':
        if 'enterocyte' in cell_type:
            return 'Enterocyte'
        elif 'paneth' in cell_type:
            return 'Paneth Cell'
        elif 'goblet' in cell_type:
            return 'Goblet Cell'
        elif 'tuft' in cell_type:
            return 'Tuft Cell'
        else:
            return 'GI Epithelial'
    
    # T cells - keep CD4/CD8 distinction
    elif broad_type == 'CD4_T_cell':
        if 'naive' in cell_type.lower():
            return 'Naive CD4 T cell'
        elif 'memory' in cell_type.lower():
            return 'Memory CD4 T cell'
        elif 'regulatory' in cell_type.lower():
            return 'Regulatory T cell'
        elif 'helper' in cell_type.lower():
            return 'Helper CD4 T cell'
        else:
            return 'CD4 T cell'
    
    elif broad_type == 'CD8_T_cell':
        if 'naive' in cell_type.lower():
            return 'Naive CD8 T cell'
        elif 'memory' in cell_type.lower():
            return 'Memory CD8 T cell'
        else:
            return 'CD8 T cell'
    
    # B cells - keep maturation state
    elif broad_type == 'B_cell':
        if 'naive' in cell_type.lower():
            return 'Naive B cell'
        elif 'memory' in cell_type.lower():
            return 'Memory B cell'
        elif 'immature' in cell_type.lower():
            return 'Immature B cell'
        else:
            return 'B cell'
    
    # Endothelial - keep vessel type
    elif 'Endothelial' in broad_type:
        if 'capillary' in cell_type.lower():
            return 'Capillary Endothelial'
        elif 'arterial' in cell_type.lower() or 'artery' in cell_type.lower():
            return 'Arterial Endothelial'
        elif 'vein' in cell_type.lower() or 'venous' in cell_type.lower():
            return 'Venous Endothelial'
        elif 'lymphatic' in cell_type.lower():
            return 'Lymphatic Endothelial'
        else:
            return 'Endothelial Cell'
    
    # Fibroblasts - distinguish organ-specific
    elif 'Fibroblast' in broad_type:
        if 'adventitial' in cell_type.lower():
            return 'Adventitial Fibroblast'
        elif 'alveolar' in cell_type.lower():
            return 'Alveolar Fibroblast'
        elif 'cardiac' in cell_type.lower():
            return 'Cardiac Fibroblast'
        elif 'myofibroblast' in cell_type.lower():
            return 'Myofibroblast'
        else:
            return 'Fibroblast'
    
    # Epithelial - keep organ context
    elif broad_type == 'Respiratory_Epithelial':
        if 'ciliated' in cell_type.lower():
            return 'Ciliated Respiratory Epithelial'
        elif 'goblet' in cell_type.lower():
            return 'Respiratory Goblet Cell'
        elif 'basal' in cell_type.lower():
            return 'Respiratory Basal Cell'
        else:
            return 'Respiratory Epithelial'
    
    elif broad_type == 'Alveolar_cell':
        if 'type i' in cell_type.lower():
            return 'Type I Pneumocyte'
        elif 'type ii' in cell_type.lower():
            return 'Type II Pneumocyte'
        else:
            return 'Alveolar Cell'
    
    # For everything else, return the broad type
    else:
        return broad_type

def clean_tissue_names(tissue):
    """
    Standardize and clean tissue names
    """
    tissue_mapping = {
        # Brain regions - group by major area
        'MTG': 'Brain_MTG',
        'A1C': 'Brain_A1C',
        'V1C': 'Brain_V1C',
        'CgG': 'Brain_CgG',
        'S1lm': 'Brain_S1lm',
        'S1ul': 'Brain_S1ul',
        'M1lm': 'Brain_M1lm',
        'M1ul': 'Brain_M1ul',
        
        # Clean up existing names
        'Large_Intestine': 'Colon',
        'Small_Intestine': 'Small_Intestine',
        'Bone_Marrow': 'Bone_Marrow',
        'Lymph_Node': 'Lymph_Node',
        'Salivary_Gland': 'Salivary_Gland',
        
        # Single word tissues are fine
        'Blood': 'Blood',
        'Muscle': 'Muscle',
        'Spleen': 'Spleen',
        'Skin': 'Skin',
        'Lung': 'Lung',
        'Liver': 'Liver',
        'Kidney': 'Kidney',
        'Heart': 'Heart',
        'Bladder': 'Bladder',
        'Fat': 'Adipose',
        'Eye': 'Eye',
        'Tongue': 'Tongue',
        'Thymus': 'Thymus',
        'Trachea': 'Trachea',
        'Vasculature': 'Vasculature',
        'Mammary': 'Mammary_Gland',
        'Prostate': 'Prostate',
        'Uterus': 'Uterus'
    }
    
    return tissue_mapping.get(tissue, tissue)

# Apply the transformations
print("Creating medium_cell_type column...")
splice_adata.obs['medium_cell_type'] = splice_adata.obs.apply(create_medium_cell_type, axis=1)

print("Cleaning tissue names...")
splice_adata.obs['tissue_clean'] = splice_adata.obs['tissue'].apply(clean_tissue_names)

# Verify the new columns
print("\n=== Medium cell type distribution (top 30) ===")
print(splice_adata.obs['medium_cell_type'].value_counts().head(30))

print("\n=== Tissue distribution ===")
print(splice_adata.obs['tissue_clean'].value_counts())

print("\n=== Example comparison ===")
sample = splice_adata.obs[['tissue', 'tissue_clean', 'cell_type', 'medium_cell_type', 'broad_cell_type']].head(20)
print(sample.to_string())

# Save the updated distribution
combined_counts = splice_adata.obs[['tissue_clean', 'medium_cell_type', 'broad_cell_type']].value_counts()
combined_counts.to_csv("cell_type_distribution_medium.csv")
print("\nSaved detailed distribution to: cell_type_distribution_medium.csv")

In [ ]:
# ----------------------------------------------------------------
print("=== RE-SUBSETTING AND VALIDATING FINAL ALIGNMENT ===")

# Step 1: Realign cells — ensure identical cell set in both
common_cells = ge_adata.obs_names.intersection(splice_adata.obs_names)
print(f"Re-aligning to {len(common_cells)} common cells")
ge_adata = ge_adata[common_cells].copy()
splice_adata = splice_adata[common_cells].copy()
assert np.all(ge_adata.obs_names == splice_adata.obs_names)

# Step 2: Filter again for cell types with ≥50 cells
cell_type_counts = ge_adata.obs['broad_cell_type'].value_counts()
valid_cell_types = cell_type_counts[cell_type_counts >= 50].index
print(f"Cell types retained after final filtering: {len(valid_cell_types)}")

cell_mask = ge_adata.obs['broad_cell_type'].isin(valid_cell_types)
ge_adata = ge_adata[cell_mask].copy()
splice_adata = splice_adata[cell_mask].copy()
print(f"✓ Retained {ge_adata.n_obs} cells after final broad cell type filtering")

# Step 3: Recheck shared genes
shared_genes = set(ge_adata.var["gene_name"]).intersection(set(splice_adata.var["gene_name"]))
print(f"✓ {len(shared_genes)} genes shared between gene expression and splicing data")

ge_adata = ge_adata[:, ge_adata.var["gene_name"].isin(shared_genes)].copy()
splice_adata = splice_adata[:, splice_adata.var["gene_name"].isin(shared_genes)].copy()

# Final sanity checks
assert np.all(ge_adata.obs_names == splice_adata.obs_names)
print("✓ Final alignment confirmed between gene expression and splicing datasets")

In [ ]:
print("=== FINAL DATA SUMMARY ===")
print(f"Final number of unique cells in splice_adata: {splice_adata.n_obs}")
print(f"Final number of unique junctions in splice_adata: {splice_adata.n_vars}")
print(f"Final number of unique ATSEs in splice_adata: {splice_adata.var['event_id'].nunique()}")
print(f"Final number of unique genes in splice_adata: {splice_adata.var['gene_name'].nunique()}")

print(f"Final number of unique cells in ge_adata: {ge_adata.n_obs}")
print(f"Final number of unique genes in ge_adata: {ge_adata.n_vars}")

In [ ]:
assert np.all(ge_adata.obs_names == splice_adata.obs_names)

In [ ]:
splice_adata.obs["total_junction_reads"].min()

In [ ]:
import os
from datetime import datetime

# Set new model date and shared output directory
new_model_date = "102025"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
GE_OUTPUT_DIR = f"/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/{new_model_date}"
os.makedirs(GE_OUTPUT_DIR, exist_ok=True)

# ---- Gene Expression ----
ge_new_filename = f"aligned_gene_expression_data_{timestamp}.h5ad"
GE_OUTPUT_PATH = os.path.join(GE_OUTPUT_DIR, ge_new_filename)

# Save filtered gene expression data
ge_adata.write(GE_OUTPUT_PATH, compression="lzf")
print(f"Filtered gene expression AnnData written to:\n{GE_OUTPUT_PATH}")

# ---- Splicing Data ----
splice_new_filename = f"aligned_splicing_data_{timestamp}.h5ad"
SPLICE_OUTPUT_PATH = os.path.join(GE_OUTPUT_DIR, splice_new_filename)
# Subset and save matched splicing data
splice_adata.write(SPLICE_OUTPUT_PATH, compression="lzf")
print(f"Filtered splicing AnnData written to:\n{SPLICE_OUTPUT_PATH}")

In [ ]:
# Save also text file with just final list of cell ids 
with open(os.path.join(GE_OUTPUT_DIR, f"filtered_cell_ids.txt_{timestamp}"), "w") as f:
    for cell_id in ge_adata.obs_names:
        f.write(f"{cell_id}\n")